# NOURA EL KHOLTI

## XGBoost pour la régression

### 1. Fondements Mathématiques

L'objectif de XGBoost en régression est de minimiser l'erreur quadratique tout en contrôlant la complexité via une régularisation ($\lambda$). Pour chaque nœud, nous calculons les métriques suivantes:
- Le Gradient ($g_i$): La différence entre la prédiction actuelle et la cible réelle ($y_{pred} - y_{real}$).
- Le Hessien ($h_i$): La dérivée seconde de l'erreur (pour l'erreur quadratique, $h_i = 1$).
- Score de Similarité: Mesure la qualité d'un regroupement de résidus dans un nœud:
$$\text{Similarité} = \frac{(\sum g_i)^2}{\sum h_i + \lambda}$$
- Gain : Détermine si une division (split) est efficace:
$$\text{Gain} = \text{Sim}_{\text{gauche}} + \text{Sim}_{\text{droite}} - \text{Sim}_{\text{racine}}$$
- Valeur de Sortie d'une Feuille:
$$\text{Sortie} = -\frac{\sum g_i}{\sum h_i + \lambda}$$

### 2. Préparation des données

Nous allons utiliser un petit jeu de données représentant une relation linéaire simple avec un peu de bruit (par exemple, la relation entre des heures d'étude et une note d'examen).

In [ ]:
import numpy as np
import pandas as pd

# données d'entraînement
X = np.array([1, 2, 3, 4, 5, 6, 7, 8]).reshape(-1, 1)
y = np.array([1.5, 3.8, 6.7, 9.0, 11.2, 13.6, 16.1, 18.5])

data = pd.DataFrame(X, columns=['Heures_Etude'])
data['Note'] = y
print(data)

```
   Heures_Etude  Note
0             1   1.5
1             2   3.8
2             3   6.7
3             4   9.0
4             5  11.2
5             6  13.6
6             7  16.1
7             8  18.5
```

### 3. Structure de l'Arbre (Base Learner)

Chaque itération du XGBoost ajoute un arbre de décision qui tente de corriger les erreurs des arbres précédents en minimisant la structure suivante :

In [ ]:
class Node:
    def __init__(self, x, gradient, hessian, indices, params):
        self.x = x
        self.gradient = gradient
        self.hessian = hessian
        self.indices = indices
        self.params = params
        self.val = self.compute_weight()
        self.left = None
        self.right = None
        self.split_feature = None
        self.split_val = None
        
        if len(indices) > params['min_samples_split']:
            self.find_best_split()

    def compute_weight(self):
        # Formule de calcul du poids de la feuille : -Sum(g) / (Sum(h) + lambda)
        return -np.sum(self.gradient[self.indices]) / (np.sum(self.hessian[self.indices]) + self.params['reg_lambda'])

    def find_best_split(self):
        # Logique de recherche du meilleur split pour diviser le nœud
        best_gain = 0
        for feature in range(self.x.shape[1]):
            x_feat = self.x[self.indices, feature]
            for val in np.unique(x_feat):
                left_idx = self.indices[x_feat <= val]
                right_idx = self.indices[x_feat > val]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                
                gain = self.compute_gain(left_idx, right_idx)
                if gain > best_gain:
                    best_gain = gain
                    self.split_feature = feature
                    self.split_val = val
                    
        if self.split_feature is not None:
            self.left = Node(self.x, self.gradient, self.hessian, 
                             self.indices[self.x[self.indices, self.split_feature] <= self.split_val], self.params)
            self.right = Node(self.x, self.gradient, self.hessian, 
                              self.indices[self.x[self.indices, self.split_feature] > self.split_val], self.params)

    def compute_gain(self, left_idx, right_idx):
        # Calcul du gain pour déterminer la qualité d'une division
        def score(idx):
            return np.square(np.sum(self.gradient[idx])) / (np.sum(self.hessian[idx]) + self.params['reg_lambda'])
        
        return 0.5 * (score(left_idx) + score(right_idx) - score(self.indices))

### 4. Implémentation du XGBoost Regressor

In [ ]:
class XGBoostRegressor:
    def __init__(self, n_estimators=10, learning_rate=0.1, reg_lambda=1, min_samples_split=2):
        self.params = {
            'n_estimators': n_estimators,
            'learning_rate': learning_rate,
            'reg_lambda': reg_lambda,
            'min_samples_split': min_samples_split
        }
        self.trees = []
        self.base_pred = None

    def fit(self, x, y):
        self.base_pred = np.mean(y)
        current_predictions = np.full(y.shape, self.base_pred)
        
        for _ in range(self.params['n_estimators']):
            # Calcul des résidus (Gradients et Hessiennes)
            gradients = current_predictions - y
            hessians = np.ones_like(y)
            
            # Entraînement d'un nouvel arbre sur les résidus
            tree = Node(x, gradients, hessians, np.arange(len(y)), self.params)
            self.trees.append(tree)
            
            # Mise à jour des prédictions
            update = self.predict_tree(x, tree)
            current_predictions += self.params['learning_rate'] * update

    def predict_tree(self, x, tree):
        return np.array([self._predict_row(row, tree) for row in x])

    def _predict_row(self, row, node):
        if node.left is None:
            return node.val
        if row[node.split_feature] <= node.split_val:
            return self._predict_row(row, node.left)
        return self._predict_row(row, node.right)

    def predict(self, x):
        predictions = np.full(x.shape[0], self.base_pred)
        for tree in self.trees:
            predictions += self.params['learning_rate'] * self.predict_tree(x, tree)
        return predictions

### 5. Résultats et Évaluation

Nous testons maintenant le modèle sur notre nouvel exemple.

In [ ]:
# Initialisation et entraînement
model = XGBoostRegressor(n_estimators=20, learning_rate=0.3)
model.fit(X, y)

# Prédiction sur les données d'entraînement
predictions = model.predict(X)

# Comparaison
results = pd.DataFrame({
    'X (Heures)': X.flatten(),
    'Réel (Note)': y,
    'Prédiction': np.round(predictions, 2)
})

print(results)

# Calcul de l'erreur finale
mse_final = np.mean((y - predictions)**2)
print(f"\nMSE Final: {mse_final:.4f}")

```
   X (Heures)  Réel (Note)  Prédiction
0           1          1.5        1.85
1           2          3.8        3.81
2           3          6.7        6.72
3           4          9.0        9.01
4           5         11.2       11.19
5           6         13.6       13.59
6           7         16.1       16.03
7           8         18.5       18.17

MSE Final: 0.0294
```

Ce modèle montre comment le boosting ajuste itérativement les prédictions. En changeant les exemples, on observe que le modèle s'adapte à la pente des données $y \approx 2.3x$. L'ajout d'arbres réduit progressivement le résidu jusqu'à obtenir une courbe qui colle aux données initiales.